# Welcome to Noah's Interactive Fitter!

# THIS NOAH's Version, do not touch, use the other one

In [89]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, Slider
from datetime import datetime

In [90]:
# function that loads the data, from the csv generated by the 
def readfile(filename):
        # read the CSV file into a dataframe (i.e. like a Python spreadsheet)
    df = pd.read_csv(filename)
    
    # convert dataframe into a 2D Numpy array
    a = df.to_numpy()
    
    # the first column are the counters
    counter_array = a[:,0]
    
    # the second column are the time stamps
    time_array = a[:,1]
    
    # the second column are the time stamps
    # arduino_time_stamp_array = a[:,2]
    
    # the third column are your measurements: either pulseTime or distance depending on how you modified the code above
    temperature_array = a[:,2]
    rh_array = a[:,3]

    return (time_array, temperature_array, rh_array)
    

In [91]:
# this function takes eps, kc, start time and start temp as parameters and runs the euler simulation 
# It is called every time these values are changed by a slider
def euler(eps, kc, initalTime, initalTemp, final_time): 

    # before you copy-paste your constants, note the changes I have made
    
    # https://illingcompany.com/product/12oz-standard-355ml-202x211-brite-can/?srsltid=AfmBOorGi1H67og6nzUoUixZYzyuDW54VpLvHtOzYKGDkJxL2sChPYh_
    
    # Rough dimensions of can
    pi = 3.142
    r = 66.25 / 2  # eff radius
    h = 122.25  # height
    
    # estimating exposed surface based on cylinder
    
    A = (2 * (pi * (r **2) )) + (h * (2 * pi * r))   # exposed surface
    V = 355 * (1/(100^3)) # volume
    rhoair = 1.2  # approx kg/m^3  ROUGH (depends on T)
    
    # https://kg-m3.com/material/aluminum
    
    rhoAl = 2712 # kg/m3  # density Al
    
    # mass of the air is equal to desnity of air * volume of air
    mair = rhoair * V # kg air
    mAl = 13 * (1/1000) # NOAH REMEMBER TO WEIGH THE CAN TO GET THIS PREISE # mass of can kg
    d = 0.095  # rough thickness of Al  
    
    # https://www.engineeringtoolbox.com/thermal-conductivity-metals-d_858.html
    k = 237 # conductivity of Al
    # print(mair,d)
    
    # https://www.engineeringtoolbox.com/specific-heat-capacity-d_391.html
    # go back to make sure this is correct
    cair = 11005 # air J/kg/K
    cAl = 897 # Al J/kg/K
    heatcap = mAl*cAl + mair*cair
    
    Tcel = 273.15 # to convert C to K
    Tamb = 23.2 + Tcel # ambient temp (K)
    T0   = initalTemp + Tcel   # starting temperature
    
    sig = 5.67e-8  # Stefan-Boltzmann W/m^2/K^4
    
    n = 0
    dt = 0.1 # time step (s)
    # If your time step is very low, it will take a long time for the euler simulation to complete, causing the application to stutter. 
    # Try a large-ish time first ~0.1, then go small once you found values you liked. 
    tn = initalTime
    Tn = T0
    
    t = []
    T = []

    t = []
    T = []

    while tn < final_time:
        dHc = kc * A * (Tn-Tamb) * dt
        dHr = eps * sig * A * (Tn**4 - Tamb**4) * dt
        Tn = Tn - (dHc + dHr)/heatcap
        tn = tn + dt
        
        delt = (dHc + dHr)*d/k/A/dt  # temp drop across wall of can - check it is tiny
    #    print(delt)
        t.append(tn)     # store the current time
        T.append(Tn-Tcel)     # store the current temperature
        # print(tn,Tn-Tcel)

    return (t, T)

In [92]:
# # this calculated the chi squared goodness of fit score between two series
# # This is my own herbs and spices, be careful, do not cite this metric, only use it to help you determine what you need

# https://en.wikipedia.org/wiki/Chi-squared_test

# The chi-squared test is asymmetric, so I might have implemented it on the wrong variable, I am not a stats guy
# euler data first, (I didn't want to code up the edge case where you have no values to interpolate)
def chiSquared(x1s, y1s, x2s, y2s): 
    # since they are not aligned on the time axis, we have to do some interpolation. 
    # (the time axis for the experimental temperatures varies, so even if we matched them, they wouldn't fit)

    # I have chosen to do the interpolation on the euler side, since we can generate more euler points than measured points reducing compute time, but you can always flip it

    sum = 0
    for x1, y1 in zip(x1s, y1s): 
        actual_value = y1 # here euler is first, so we are assuming it is the actual and our data is observed, again might be wrong
        # computer slows when flipped, it seems numpy array ops are slow when doing many thousands of array comparisons, 
        # I don't feel like writing something faster, and this works,

        point_after_index = np.searchsorted(x2s, x1)
        point_before_index = point_after_index - 1


        if point_after_index >= len(y2s):
            # we are at the end of the list, so just use the last interpolation  
            point_after_index -= 1
            point_before_index -= 1

        m = (y2s[point_after_index] - y2s[point_before_index])/ (x2s[point_after_index] - x2s[point_before_index])

        interpolated_value = (m * (x1 - x2s[point_before_index])) + y2s[point_before_index]

        sum += ((interpolated_value -  actual_value)**2) / (actual_value)

    return sum

# test case, should be 0.0
a = np.array
assert chiSquared(a([0.5, 1.5, 2.5]), a([1, 3, 5]), a([0, 1, 2, 3]), a([0, 2, 4, 6])) == 0.0
chiSquared(a([0.5, 1.5, 2.5]), a([1, 3, 5]), a([0, 1, 2]), a([0, 2, 4]))


0.0

The following generates a matplotlib widget that will help fit your code. 

It generates the widget in a new window by default, but if you are having issues, with this, comment out the line:

%matplotlib qt

and uncomment the line 

%matplotlib ipympl


Then run the code in the physics lab server. This will put the chart in the jupyter notebook itself and my resolve some issues.

All else fails, contact me. 

I recommend putting the window in fullscreen mode. You can also zoom in with the zoom feature by selecting a portion of the graph. 

In [93]:

%matplotlib qt

# %matplotlib ipympl
# uncomment this if problems ^^ and run in the physics lab server (has to do with dependancies and GUI toolkits)

# https://matplotlib.org/stable/gallery/widgets/slider_demo.html

time_array, temperature_array, rh_array = readfile("/Users/sam/Desktop/Science One For Real/physics/SCIE001Physics/dataSorting/bestRuns/pretty good.csv")

t = time_array

# Define initial parameters
# CHANGE THIS, this worked for me, but I probably put wrong constants so you will most likely need to change your initial values
init_eps = 0.002
init_kc = 0.001

init_temp = 35
init_time = 230

# Create the figure and the line that we will manipulate
fig, ax = plt.subplots()
# ax.set_xlabel('Time [s]')

color = 'tab:blue'
ax.plot(time_array, temperature_array, marker='o', linestyle='--', color=color, ms=2)
ax.set_ylabel("Temperature ($^\\circ C$)", color=color)
# ax.set_ylim([15,50])
ax.grid(visible=True, axis='x')
ax.tick_params(axis='y', labelcolor=color)
ax.set_xlabel('time (s)')

t, T = euler(init_kc, init_eps, init_time, init_temp, time_array[-1])
line, = ax.plot(t, T, lw=2, color='tab:red')

chi_text = ax.text(0.5, 0.5, f'chi-squared {chiSquared(t, T, time_array, temperature_array)}', size=15, transform=ax.transAxes, ha='left', va='top')

# adjust the main plot to make room for the sliders
fig.subplots_adjust(left=0.25, bottom=0.25)

axfreq = fig.add_axes([0.25, 0.15, 0.65, 0.03])
kc_slider = Slider(
    ax=axfreq,
    label='kc',
    valmin=0.0, # change these if your kc is out of this range, (totally ok)
    valmax=0.002,
    valinit=init_kc,
)

axamp = fig.add_axes([0.1, 0.25, 0.0225, 0.63])
eps_slider = Slider(
    ax=axamp,
    label="eps",
    valmin=0.0,
    valmax=0.02, # change these if your eps is out of this range
    valinit=init_eps,
    orientation="vertical"
)

axTimeStart = fig.add_axes([0.25, 0.1, 0.65, 0.03])
timeStart_slider = Slider(
    ax=axTimeStart,
    label='timeStart',
    valmin=min(time_array),
    valmax=max(time_array),
    valinit=init_time,
)

axTempStart = fig.add_axes([0.05, 0.25,  0.03, 0.65])
tempStart_slider = Slider(
    ax=axTempStart,
    label='tempStart',
    valmin=min(temperature_array),
    valmax=max(temperature_array),
    valinit=init_temp,
    orientation="vertical"
)

sliders = [kc_slider, eps_slider, timeStart_slider, tempStart_slider]


# The function to be called anytime a slider's value changes
def update(val):
    t, T = euler(kc_slider.val, eps_slider.val, timeStart_slider.val, tempStart_slider.val, time_array[-1])
    line.set_ydata(T)
    line.set_xdata(t)
    chi_text.set_text(f"chi-squared {chiSquared(t, T, time_array, temperature_array)}")

    fig.canvas.draw_idle()

for slider in sliders: 
    # register the update function with each slider
    slider.on_changed(update)


# Create a `matplotlib.widgets.Button` to reset the sliders to initial values.
resetax = fig.add_axes([0.8, 0.025, 0.05, 0.04])
button = Button(resetax, 'Reset', hovercolor='0.975')

def reset(event):
    for slider in sliders: 
        slider.reset()
    
button.on_clicked(reset)

# Create a `matplotlib.widgets.Button` to reset the sliders to initial values.
resetax2 = fig.add_axes([0.5, 0.025, 0.2, 0.04])
printButton = Button(resetax2, 'Print Numbers in Notebook', hovercolor='0.975')

def printnumbers(event):
    print(f"time={datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  ", end="")
    for slider in sliders: 
        print(f"{slider.label.get_text()} = {slider.val} \t", end="")

    print(f"chi_2 = {chi_text.get_text()} \t", end="")
    print()
    
printButton.on_clicked(printnumbers)

plt.show()